In [1]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
zip_path = '/content/drive/MyDrive/! School/Sem 8/AI Agents/topic5/Corpora.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/')

print("✓ Corpora extracted to /content/Corpora")

Mounted at /content/drive
✓ Corpora extracted to /content/Corpora


# Manual RAG Pipeline: Mechanisms First

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline from scratch.
You'll see every step explicitly before we move to frameworks like LangChain.

**Works on:** Google Colab, Local Jupyter (Mac/Windows/Linux)

**Pipeline Overview:**
```
Documents → Chunking → Embedding → Index (FAISS)
                                        ↓
User Query → Embed Query → Similarity Search → Top-K Chunks
                                                    ↓
                                        Prompt Assembly → LLM → Answer
```

## TODO — Topic 5 RAG Course Project Checklist

- **Exercise 0:** Set-up — Get notebook running; unzip Corpora.zip. Use PDFs from `Corpora/<corpus>/pdf_embedded/`.
- **Exercise 1:** Open model RAG vs no RAG — Compare Qwen 2.5 1.5B with/without RAG on Model T manual and Congressional Record.
- **Exercise 2:** Open model + RAG vs large model — Run GPT-4o Mini with no tools on same queries.
- **Exercise 3:** Open model + RAG vs frontier chat — Compare local Qwen+RAG vs GPT-4/Claude (web).
- **Exercise 4:** Effect of top-K — Test k = 1, 3, 5, 10, 20.
- **Exercise 5:** Unanswerable questions — Off-topic, related-but-missing, false premise.
- **Exercise 6:** Query phrasing sensitivity — Same question in 5+ phrasings.
- **Exercise 7:** Chunk overlap — Re-chunk with overlap 0, 64, 128, 256.
- **Exercise 8:** Chunk size — Chunk at 128, 256, 512, 1024, 2048.
- **Exercise 9:** Retrieval score analysis — 10 queries, top-10 chunks, score distribution.
- **Exercise 10:** Prompt template variations — Minimal, strict grounding, citation, permissive, structured.
- **Exercise 11:** Failure mode catalog — Computation, temporal, comparison, ambiguous, multi-hop, etc.
- **Exercise 12:** Cross-document synthesis — Questions needing multiple chunks.

## Setup

First, let's install the required packages and detect our compute environment.

In [21]:
# Install dependencies
# On Colab, these install quickly. Locally, you may already have them.
# Use a kernel-aware install when available; fall back to subprocess otherwise.
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q torch transformers sentence-transformers faiss-cpu pymupdf accelerate ipyfilechooser')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'transformers', 'sentence-transformers', 'faiss-cpu', 'pymupdf', 'accelerate', 'ipyfilechooser'])
# For Exercise 2 (GPT-4o Mini): add 'openai' to the list above if needed


In [22]:
# =============================================================================
# ENVIRONMENT AND DEVICE DETECTION
# =============================================================================
import os
import sys

# Enable MPS fallback for any PyTorch operations not yet implemented on Metal
# This MUST be set before importing torch
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Prevent kernel crash from duplicate OpenMP libraries (PyTorch + FAISS conflict on macOS)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from typing import Tuple

def detect_environment() -> str:
    """Detect if we're running on Colab or locally."""
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

def get_device() -> Tuple[str, torch.dtype]:
    """
    Detect the best available compute device.

    Priority: CUDA > MPS (Apple Silicon) > CPU

    Returns:
        Tuple of (device_string, recommended_dtype)

    Notes:
        - CUDA: Use float16 for memory efficiency (Tensor Cores optimize this)
        - MPS: Use float32 - Apple Silicon doesn't have the same float16
               optimizations as NVIDIA, and float32 is often faster
        - CPU: Use float32 (float16 not well supported on CPU)
    """
    if torch.cuda.is_available():
        device = 'cuda'
        dtype = torch.float16
        device_name = torch.cuda.get_device_name(0)
        memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ Using CUDA GPU: {device_name} ({memory_gb:.1f} GB)")

    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        device = 'mps'
        dtype = torch.float32  # float32 is often faster on Apple Silicon!
        print("✓ Using Apple Silicon GPU (MPS)")
        print("  Note: Using float32 (faster than float16 on Apple Silicon)")

    else:
        device = 'cpu'
        dtype = torch.float32
        print("⚠ Using CPU (no GPU detected)")
        print("  Tip: For faster processing, use a machine with a GPU")

    return device, dtype

# Detect environment and device
ENVIRONMENT = detect_environment()
DEVICE, DTYPE = get_device()

print(f"\nEnvironment: {ENVIRONMENT.upper()}")
print(f"Device: {DEVICE}, Dtype: {DTYPE}")

✓ Using CUDA GPU: Tesla T4 (15.6 GB)

Environment: COLAB
Device: cuda, Dtype: torch.float16


## Load Your Documents

**Cell 1:** Configure your document source and select/upload files
- **Local Jupyter**: Use the folder picker, then run Cell 2
- **Colab + Upload**: Files upload immediately (blocking), then run Cell 2
- **Colab + Drive**: Set `USE_GOOGLE_DRIVE = True`, mounts Drive and shows picker, then run Cell 2

**Cell 2:** Confirms selection and lists documents

In [49]:
# =============================================================================
# CELL 1: SELECT DOCUMENT SOURCE
# =============================================================================
from pathlib import Path

# ------------- COLAB USERS: CONFIGURE HERE -------------
USE_GOOGLE_DRIVE = False  # Set to True to use Google Drive instead of uploading
# -------------------------------------------------------

_folder_default = Path("/content/Corpora/ModelTService")
DOC_FOLDER = "/content/Corpora/NewModelT"
folder_chooser = None

if ENVIRONMENT == 'colab':
    if USE_GOOGLE_DRIVE:
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
        print("✓ Google Drive mounted\n")
        try:
            from ipyfilechooser import FileChooser
            folder_chooser = FileChooser(
                path='/content/drive/MyDrive',
                title='Select your documents folder in Google Drive',
                show_only_dirs=True,
                select_default=True
            )
            print("📁 Select your documents folder below, then run Cell 2:")
            display(folder_chooser)
        except ImportError:
            print("Folder picker not available.")
            DOC_FOLDER = '/content/drive/MyDrive/your_documents_folder'
            print(f"  DOC_FOLDER = '{DOC_FOLDER}'")
    else:
        if Path(DOC_FOLDER).exists():
            print(f"✓ Using existing folder: {DOC_FOLDER}")
        else:
            from google.colab import files
            os.makedirs(DOC_FOLDER, exist_ok=True)
            print("Upload your documents (PDF, TXT, or MD):")
            uploaded = files.upload()
            for filename in uploaded.keys():
                os.rename(filename, f'{DOC_FOLDER}/{filename}')
                print(f"  ✓ Saved: {DOC_FOLDER}/{filename}")
        print(f"\n✓ Ready. Run Cell 2 to continue.")

else:
    print("Running locally\n")
    try:
        from ipyfilechooser import FileChooser
        folder_chooser = FileChooser(
            path=str(Path.home()),
            title='Select your documents folder',
            show_only_dirs=True,
            select_default=True
        )
        print("📁 Select your documents folder below, then run Cell 2:")
        display(folder_chooser)
    except ImportError:
        print("Folder picker not available (ipyfilechooser not installed).")
        print(f"\nUsing default folder: {Path(DOC_FOLDER).absolute()}")
        os.makedirs(DOC_FOLDER, exist_ok=True)

✓ Using existing folder: /content/Corpora/NewModelT

✓ Ready. Run Cell 2 to continue.


In [50]:
# =============================================================================
# CELL 2: CONFIRM SELECTION AND LIST DOCUMENTS
# =============================================================================
# If you used a folder picker above, make sure you selected a folder
# BEFORE running this cell. The picker is non-blocking.
# =============================================================================

# Read selection from folder picker (if one was used)
if folder_chooser is not None and folder_chooser.selected_path:
    DOC_FOLDER = folder_chooser.selected_path
    print(f"✓ Using selected folder: {DOC_FOLDER}")
elif folder_chooser is not None:
    print("⚠ No folder selected in picker!")
    print("  Please go back to Cell 1, select a folder, then run this cell again.")
else:
    # No picker used (upload or manual path)
    print(f"✓ Using folder: {DOC_FOLDER}")

# Confirm folder (listing skipped for speed)
doc_path = Path(DOC_FOLDER)
if doc_path.exists():
    print(f"✓ Folder set: {doc_path.absolute()}")
    print("  Run the next cells to load, chunk, and index documents.")
else:
    print(f"⚠ Folder not found: {DOC_FOLDER}")
    print("  Please set DOC_FOLDER in the previous cell and run it again.")

✓ Using folder: /content/Corpora/NewModelT
✓ Folder set: /content/Corpora/NewModelT
  Run the next cells to load, chunk, and index documents.


---
## Stage 1: Document Loading

We need to extract text from our documents. For PDFs with embedded text,
PyMuPDF (fitz) reads the text layer directly - no OCR needed.

**Corpora:** Use PDFs from `Corpora/<name>/pdf_embedded/`. The `.txt` files in `txt/` are for checking retrieval vs OCR issues.

In [51]:
# Exercise 1 (and reuse): Official query lists. Reference: CR Jan 13, 20, 21, 23, 2026.
QUERIES_MODEL_T = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?",
]
QUERIES_CR = [
    "What did Mr. Flood have to say about Mayor David Black in Congress on January 13, 2026?",
    "What mistake did Elise Stefanik make in Congress on January 23, 2026?",
    "What is the purpose of the Main Street Parity Act?",
    "Who in Congress has spoken for and against funding of pregnancy centers?",
]

In [52]:
import fitz  # PyMuPDF
from typing import List, Tuple

def load_text_file(filepath: str) -> str:
    """Load a plain text file."""
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()


def load_pdf_file(filepath: str) -> str:
    """
    Extract text from a PDF with embedded text.

    PyMuPDF reads the text layer directly.
    For scanned PDFs without embedded text, you'd need OCR.
    """
    doc = fitz.open(filepath)
    text_parts = []

    for page_num, page in enumerate(doc):
        text = page.get_text()
        if text.strip():
            # Add page marker for debugging/citation
            text_parts.append(f"\n[Page {page_num + 1}]\n{text}")

    doc.close()
    return "\n".join(text_parts)


def load_documents(doc_folder: str) -> List[Tuple[str, str]]:
    """Load all documents from a folder. Returns list of (filename, content)."""
    documents = []
    folder = Path(doc_folder)

    for filepath in folder.rglob("*"):
        try:
            if not filepath.is_file():
                continue
        except OSError:
            continue
        if filepath.suffix.lower() not in ('.pdf', '.txt', '.md', '.text'):
            continue
        try:
            if filepath.suffix.lower() == '.pdf':
                content = load_pdf_file(str(filepath))
            elif filepath.suffix.lower() in ['.txt', '.md', '.text']:
                content = load_text_file(str(filepath))
            else:
                continue

            if content.strip():
                documents.append((filepath.name, content))
                print(f"✓ Loaded: {filepath.name} ({len(content):,} chars)")
        except Exception as e:
            print(f"✗ Error loading {filepath}: {e}")

    return documents

In [53]:
# Load your documents
documents = load_documents(DOC_FOLDER)
print(f"\nLoaded {len(documents)} documents")

if len(documents) == 0:
    print("\n⚠ No documents loaded! Please add PDF or TXT files to the documents folder.")

✓ Loaded: ModelTNew.txt (545,492 chars)
✓ Loaded: ModelTNew.pdf (469,891 chars)

Loaded 2 documents


In [54]:
# Inspect a document to verify loading worked
if documents:
    filename, content = documents[0]
    print(f"First document: {filename}")
    print(f"Total length: {len(content):,} characters")
    print(f"\nFirst 1000 characters:\n{'-'*40}")
    print(content[:1000])

First document: ModelTNew.txt
Total length: 545,492 characters

First 1000 characters:
----------------------------------------
SERVI

 Detailed Instructions for
  Servicing Ford Gars




    PRICE $250



         Published by




 DETROIT, MICHIGAN, U. S. A.
                                         Contents

Foreword . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .    111
Essentials of good service. . . . : . . . . . : . . . . . . . . . . . . . . . . . . . . . . .               ix
Ideal shop layout for average size dealer. . . . . . . . . . . . . . . . . . . . .                           x
Essential shop equipment. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .                  xi
The parts department. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
                                                                                                            ...
                                   

---
## Stage 2: Chunking

Documents need to be split into pieces small enough to be relevant but large enough to carry meaning.

**Why overlap?** If a key sentence sits right at a chunk boundary, splitting without overlap might cut it in half. Overlap ensures that information near boundaries appears intact in at least one chunk.

**Experiment:** Try different chunk sizes (256, 512, 1024) and see how it affects retrieval!

In [55]:
from dataclasses import dataclass

@dataclass
class Chunk:
    """A chunk of text with metadata for tracing back to source."""
    text: str
    source_file: str
    chunk_index: int
    start_char: int
    end_char: int


def chunk_text(
    text: str,
    source_file: str,
    chunk_size: int = 512,
    chunk_overlap: int = 128
) -> List[Chunk]:
    """
    Split text into overlapping chunks.

    We try to break at sentence or paragraph boundaries
    to avoid cutting mid-thought.
    """
    chunks = []
    start = 0
    chunk_index = 0

    while start < len(text):
        end = start + chunk_size

        # Try to break at a good boundary
        if end < len(text):
            # Look for paragraph break first
            para_break = text.rfind('\n\n', start + chunk_size // 2, end)
            if para_break != -1:
                end = para_break + 2
            else:
                # Look for sentence break
                sentence_break = text.rfind('. ', start + chunk_size // 2, end)
                if sentence_break != -1:
                    end = sentence_break + 2

        chunk_text_str = text[start:end].strip()

        if chunk_text_str:
            chunks.append(Chunk(
                text=chunk_text_str,
                source_file=source_file,
                chunk_index=chunk_index,
                start_char=start,
                end_char=end
            ))
            chunk_index += 1

        # Move forward, accounting for overlap
        start = end - chunk_overlap
        if chunks and start <= chunks[-1].start_char:
            start = end  # Safety: ensure progress

    return chunks

In [56]:
# ============================================
# EXPERIMENT: Try different chunk sizes!
# ============================================
CHUNK_SIZE = 512      # Try: 256, 512, 1024
CHUNK_OVERLAP = 128   # Try: 64, 128, 256
# For Ex 7/8 use rebuild_pipeline() — see cell after FAISS index.

# Chunk all documents
all_chunks = []
for filename, content in documents:
    doc_chunks = chunk_text(content, filename, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(doc_chunks)
    print(f"{filename}: {len(doc_chunks)} chunks")

print(f"\nTotal: {len(all_chunks)} chunks")

ModelTNew.txt: 1781 chunks
ModelTNew.pdf: 1496 chunks

Total: 3277 chunks


In [57]:
# Inspect some chunks
if all_chunks:
    print("Sample chunks:")
    indices_to_show = [0, len(all_chunks)//2, -1] if len(all_chunks) > 2 else range(len(all_chunks))
    for i in indices_to_show:
        chunk = all_chunks[i]
        print(f"\n{'='*60}")
        print(f"Chunk {chunk.chunk_index} from {chunk.source_file}")
        print(f"{'='*60}")
        print(chunk.text[:300] + "..." if len(chunk.text) > 300 else chunk.text)

Sample chunks:

Chunk 0 from ModelTNew.txt
SERVI

 Detailed Instructions for
  Servicing Ford Gars




    PRICE $250



         Published by




 DETROIT, MICHIGAN, U. S. A.
                                         Contents

Foreword . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .    111
Ess...

Chunk 1638 from ModelTNew.txt
Fig. 569
                          FORD SERVICE                             283




                                Fig. 570

1262 Raise the seat back approximately 2", this will release the clip
  which holds seat back to spacer board. Seat back can then be lifted
  out of car as shown in Fig. 567...

Chunk 1495 from ModelTNew.pdf
tening.. . . ....... . . . . .. . .. . .. 
74 
removing . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
57 
Wiring diagram, cars equipped with starter (Page 19) 
not equipped with starter (Page 20) 
improved cars equipped with starter (Page 287) · 
not equipped 

---
## Stage 3: Embedding

Embeddings map text to dense vectors where **semantic similarity = geometric proximity**.

A sentence about "cardiac arrest" and one about "heart attack" will have similar embeddings even though they share no words.

**Note:** sentence-transformers does NOT auto-detect Apple MPS - we must pass the device explicitly.

In [58]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
# Options:
# - "sentence-transformers/all-MiniLM-L6-v2": Fast, small (80MB), good quality
# - "BAAI/bge-small-en-v1.5": Better for retrieval, similar size

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBEDDING_MODEL}")
print(f"Device: {DEVICE}")

# Must explicitly pass device for MPS support!
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDING_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {EMBEDDING_DIM}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


In [59]:
# DEMO: See how embeddings capture semantic similarity
test_sentences = [
    "The engine needs regular oil changes.",
    "Motor oil should be replaced periodically.",
    "The Senate convened at noon.",
    "Congress began its session at midday."
]

test_embeddings = embed_model.encode(test_sentences)

# Compute cosine similarity matrix
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print("Cosine similarity matrix:")
print("\n" + " " * 40 + "  [0]    [1]    [2]    [3]")
for i, s1 in enumerate(test_sentences):
    sims = [cosine_sim(test_embeddings[i], test_embeddings[j]) for j in range(4)]
    print(f"[{i}] {s1[:35]:35} {sims[0]:.3f}  {sims[1]:.3f}  {sims[2]:.3f}  {sims[3]:.3f}")

print("\n→ Notice: [0]-[1] are similar (both about oil), [2]-[3] are similar (both about Congress)")

Cosine similarity matrix:

                                          [0]    [1]    [2]    [3]
[0] The engine needs regular oil change 1.000  0.728  -0.045  -0.032
[1] Motor oil should be replaced period 0.728  1.000  0.014  0.035
[2] The Senate convened at noon.        -0.045  0.014  1.000  0.684
[3] Congress began its session at midda -0.032  0.035  0.684  1.000

→ Notice: [0]-[1] are similar (both about oil), [2]-[3] are similar (both about Congress)


In [60]:
# Embed all chunks - this may take a few minutes for large corpora
if all_chunks:
    print(f"Embedding {len(all_chunks)} chunks on {DEVICE}...")
    chunk_texts = [c.text for c in all_chunks]
    chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)
    chunk_embeddings = chunk_embeddings.astype('float32')  # FAISS wants float32
    print(f"Embeddings shape: {chunk_embeddings.shape}")
else:
    print("No chunks to embed - please load documents first.")

Embedding 3277 chunks on cuda...


Batches:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (3277, 384)


---
## Stage 4: Vector Index (FAISS)

FAISS efficiently finds nearest neighbors in high-dimensional spaces.

We use a simple **flat index** (brute-force search) which is transparent and works well for up to ~100k vectors. For larger corpora, you'd use approximate methods like IVF or HNSW.

**Note:** FAISS GPU support is CUDA-only. On MPS/CPU, we use faiss-cpu (still very fast for <100k vectors).

In [61]:
import faiss

# Create FAISS index
# IndexFlatIP = Inner Product (for cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(EMBEDDING_DIM)

if all_chunks:
    # Normalize vectors so inner product = cosine similarity
    faiss.normalize_L2(chunk_embeddings)

    # Add vectors to index
    index.add(chunk_embeddings)
    print(f"Index built with {index.ntotal} vectors")
else:
    print("No embeddings to index - please load and embed documents first.")

Index built with 3277 vectors


---
## Stage 5: Retrieval

Now we can search! Given a query, we:
1. Embed the query with the same model
2. Find the top-k most similar chunks
3. Return those chunks as context

In [62]:
# Helper for Exercises 7 & 8: rebuild chunks + index with different chunk_size / chunk_overlap.
def rebuild_pipeline(chunk_size: int = 512, chunk_overlap: int = 128):
    """Re-chunk documents, re-embed, and rebuild FAISS index. Updates global all_chunks and index."""
    global all_chunks, index
    all_chunks = []
    for filename, content in documents:
        all_chunks.extend(chunk_text(content, filename, chunk_size=chunk_size, chunk_overlap=chunk_overlap))
    chunk_embeddings = embed_model.encode([c.text for c in all_chunks], show_progress_bar=True).astype("float32")
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"Rebuilt: {len(all_chunks)} chunks, chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")

In [63]:
def retrieve(query: str, top_k: int = 5):
    """
    Retrieve the top-k most relevant chunks for a query.

    Returns: List of (chunk, similarity_score) tuples
    """
    # Embed the query
    query_embedding = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)

    # Search
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append((all_chunks[idx], float(score)))

    return results

In [64]:
# Test retrieval
# ============================================
# TRY DIFFERENT QUERIES FOR YOUR CORPUS!
# ============================================
test_query = "What is the procedure for engine maintenance?"  # ← Modify this!

if index.ntotal > 0:
    results = retrieve(test_query, top_k=5)

    print(f"Query: {test_query}\n")
    print("Top 5 retrieved chunks:")
    for i, (chunk, score) in enumerate(results, 1):
        print(f"\n[{i}] Score: {score:.4f} | Source: {chunk.source_file}")
        print(f"    {chunk.text[:200]}...")
else:
    print("Index is empty - please load, chunk, and embed documents first.")

Query: What is the procedure for engine maintenance?

Top 5 retrieved chunks:

[1] Score: 0.5769 | Source: ModelTNew.pdf
    en a car is brought in for major repair 
work is to first assign the car to a section of the shop set aside for re- 
pair jobs. The assembly to be overhauled is then removed from the 
car and by means...

[2] Score: 0.5550 | Source: ModelTNew.txt
    be performed.
    When the overhaul work is completed the assembly is returned by
means of the overhead track t o the car from. which it, was removed.
It is then installed in the car and the job is co...

[3] Score: 0.5432 | Source: ModelTNew.pdf
    cleaning. After the cleaning operation it is trans- 
ferred to the stand or repair bench on which the work is to be performed. 
When the overhaul work is completed the assembly is returned by 
means o...

[4] Score: 0.5397 | Source: ModelTNew.txt
    , install car covers, lift off hood , remove cylinder
      head and valve cover .. . . ... . . . ... . . . . .. .........

---
## Stage 6: Generation (LLM)

Now we load a local LLM to generate answers from the retrieved context.

**Recommended models:**
- `Qwen/Qwen2.5-1.5B-Instruct` - Best instruction following at this size
- `Qwen/Qwen2.5-3B-Instruct` - Even better if you have 8GB+ VRAM
- `meta-llama/Llama-3.2-1B-Instruct` - Alternative, slightly weaker

**Device handling:**
- CUDA: Uses `device_map="auto"` and float16
- MPS: Loads to CPU first, then moves to MPS with float32
- CPU: Uses float32 (slower but works)

In [39]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# ============================================
# CHOOSE YOUR MODEL
# ============================================
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # Or try "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading LLM: {LLM_MODEL}")
print(f"Device: {DEVICE}, Dtype: {DTYPE}")
print("This may take a few minutes on first run...\n")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

# Load with appropriate settings for each device type
if DEVICE == 'cuda':
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        device_map="auto",
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    print("Model loaded on CUDA")

elif DEVICE == 'mps':
    # For MPS, load to CPU first, then move to MPS
    # (device_map="auto" doesn't work well with MPS)
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    model = model.to(DEVICE)
    print("Model loaded on MPS (Apple Silicon)")

else:
    # CPU
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    print("Model loaded on CPU (this will be slow)")

Loading LLM: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda, Dtype: torch.float16
This may take a few minutes on first run...



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded on CUDA


In [40]:
def generate_response(prompt: str, max_new_tokens: int = 512, temperature: float = 0.3) -> str:
    """
    Generate a response from the LLM.

    Lower temperature = more focused/deterministic
    Higher temperature = more creative/random
    """
    inputs = tokenizer(prompt, return_tensors="pt")

    # Move inputs to the correct device
    if DEVICE == 'cuda':
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the new tokens
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()

---
## Stage 7: The Complete RAG Pipeline

Now we put it all together. The **prompt template** is critical - it must instruct the model to use the retrieved context.

In [41]:
# The RAG prompt template
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer the question based ONLY on the information in the context above
- If the context doesn't contain enough information to answer, say so
- Quote relevant parts of the context to support your answer
- Be concise and direct

ANSWER:"""


def direct_query(question: str, max_new_tokens: int = 512) -> str:
    """Ask the LLM directly with no retrieved context (for RAG vs no-RAG comparison)."""
    prompt = f"""Answer this question:
{question}

Answer:"""
    return generate_response(prompt, max_new_tokens=max_new_tokens)

def rag_query(question: str, top_k: int = 5, show_context: bool = False, prompt_template: str = None) -> str:
    """The complete RAG pipeline. prompt_template: custom template for Exercise 10."""
    # Step 1: Retrieve
    results = retrieve(question, top_k)

    # Format context
    context_parts = []
    for chunk, score in results:
        context_parts.append(f"[Source: {chunk.source_file}, Relevance: {score:.3f}]\n{chunk.text}")
    context = "\n\n---\n\n".join(context_parts)

    if show_context:
        print("=" * 60)
        print("RETRIEVED CONTEXT:")
        print("=" * 60)
        print(context)
        print("=" * 60 + "\n")

    # Step 2: Build prompt (use custom template if provided)
    template = prompt_template if prompt_template is not None else PROMPT_TEMPLATE
    prompt = template.format(context=context, question=question)

    # Step 3: Generate
    answer = generate_response(prompt)

    return answer

In [42]:
# ============================================
# TEST YOUR RAG PIPELINE!
# ============================================

question = "What maintenance is required for the engine?"  # ← Modify for your corpus!

if index.ntotal > 0:
    print(f"Question: {question}\n")
    print("Generating answer...\n")

    answer = rag_query(question, top_k=5, show_context=True)

    print("ANSWER:")
    print(answer)
else:
    print("Pipeline not ready - please complete all previous stages first.")

Question: What maintenance is required for the engine?

Generating answer...

RETRIEVED CONTEXT:
[Source: CREC-2026-01-22.pdf, Relevance: 0.445]
acility upgrades; 
(2) workforce development, training, and 
retention; 
(3) supplier base expansion and qualifica-
tion, including second- and third-tier ven-
dors and non-traditional manufacturers; 
(4) process improvements, automation, and 
digital manufacturing; and 
(5) risk reduction and surge capacity ini-
tiatives necessary to ensure reliable, afford-
able, and timely production of solid rocket 
motors and related energetics: 
(b) Not later than 60 days after the date of 
the enactment of this Act,

---

[Source: CREC-2026-01-08-bk3.pdf, Relevance: 0.395]
rects the Department to 
continue maximizing utilization of EERE- 
stewarded 
facilities 
and 
projects 
across 
EERE programs. 
The Department is encouraged to identify 
technical needs, such as biobased lubricants 
for hydropower equipment, or other mission- 
critical applications, 

---
## Experiments: Understanding RAG Behavior

Now that you have a working pipeline, try these experiments to understand how each component affects the results.

In [53]:
# EXPERIMENT 1: Compare WITH vs WITHOUT RAG
# ==========================================

question = "What are the specifications for the landing gear?"  # ← Use a corpus-specific question!

if index.ntotal > 0:
    # WITHOUT RAG - just ask the model directly
    direct_prompt = f"""Answer this question:
{question}

Answer:"""

    print("WITHOUT RAG (model's own knowledge):")
    print("-" * 40)
    direct_answer = generate_response(direct_prompt)
    print(direct_answer)

    print("\n" + "=" * 60 + "\n")

    # WITH RAG
    print("WITH RAG (using retrieved context):")
    print("-" * 40)
    rag_answer = rag_query(question, top_k=5)
    print(rag_answer)
else:
    print("Please complete the pipeline setup first.")

WITHOUT RAG (model's own knowledge):
----------------------------------------
The landing gear is a set of wheels that extend from an aircraft to provide ground contact during takeoff and landing. It consists of two main parts, the nose wheel and the main wheels.

The landing gear is designed to be flexible so it can absorb shock when the plane hits the runway or water in case of an emergency landing. The landing gear also has brakes on them which help slow down the aircraft after touchdown.
The landing gear is typically made of aluminum alloy or steel and is reinforced with rubber or other materials to protect against impact. Some modern aircraft have retractable landing gears that fold up into the fuselage when not in use, while others have fixed landing gears that remain exposed at all times. The size and weight of the landing gear depend on the type of aircraft and its intended use. For example, military transport planes may have larger and heavier landing gear than passenger jets.

In [54]:
# EXPERIMENT 2: Effect of top_k
# ==========================================

question = "What safety procedures are required?"  # ← Use a corpus-specific question!

if index.ntotal > 0:
    for k in [1, 3, 5, 10]:
        print(f"\n{'='*60}")
        print(f"TOP_K = {k}")
        print(f"{'='*60}")
        answer = rag_query(question, top_k=k)
        print(answer[:500] + "..." if len(answer) > 500 else answer)
else:
    print("Please complete the pipeline setup first.")


TOP_K = 1
The context does not provide specific safety procedures related to soldering or the use of a soldering iron. It focuses more on the technical aspects such as selecting an appropriate iron size, taper shape, and cleaning requirements. Therefore, there is insufficient information to determine what safety procedures are required according to this source. 

Relevant parts from the context:

"The iron should be tapered to a Hat point for getting into the corners."

This suggests some level of ergonom...

TOP_K = 3
The context does not provide specific safety procedures related to installing or maintaining a Model T. It focuses more on technical aspects such as selecting a suitable soldering iron and cleaning its surfaces. Therefore, there isn't enough information to determine what safety procedures might be required. 

Relevant text from the context:

"Cleanliness of the iron is most important as the solder cannot be drawn evenly unless the iron is clean." 

This statement emphas

In [55]:
# EXPERIMENT 3: Question the corpus CAN'T answer
# ==========================================
# Does the model admit it doesn't know, or hallucinate?

unanswerable_question = "What is the CEO's favorite color?"

if index.ntotal > 0:
    print(f"Question: {unanswerable_question}\n")
    answer = rag_query(unanswerable_question, top_k=5, show_context=True)
    print(f"\nAnswer: {answer}")
else:
    print("Please complete the pipeline setup first.")

Question: What is the CEO's favorite color?

RETRIEVED CONTEXT:
[Source: ModelTNew.txt, Relevance: 0.297]
TAIL
                                                                                                                                                                                                 BRIGHT
No.3Biue

---

[Source: ModelTNew.txt, Relevance: 0.281]
Cable-Switch to Terminal Block                            Switch to Coil Box - Blue with Yellow Tracer
                                                                                                 Head Lamp Wire-Black with Green Tracer                           Switch to Terminal Block !Battery-Yellow)
                                                                                                 Coil Box to Switch Wire- -Blue with Yellow

---

[Source: ModelTNew.txt, Relevance: 0.267]
to          Terminal- Yellow with Black Tracer
                                                                                          

---
## Save/Load Your Index

For large corpora, you don't want to re-embed every time. Here's how to persist the index.

In [43]:
import pickle

def save_index(filepath: str):
    """Save FAISS index and chunks to disk."""
    faiss.write_index(index, f"{filepath}.faiss")
    with open(f"{filepath}.chunks", 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f"✓ Saved index to {filepath}.faiss")
    print(f"✓ Saved chunks to {filepath}.chunks")

def load_saved_index(filepath: str):
    """Load FAISS index and chunks from disk."""
    global index, all_chunks
    index = faiss.read_index(f"{filepath}.faiss")
    with open(f"{filepath}.chunks", 'rb') as f:
        all_chunks = pickle.load(f)
    print(f"✓ Loaded index with {index.ntotal} vectors")

# Save your index
if index.ntotal > 0:
    save_index("my_rag_index")
else:
    print("No index to save.")

# Later, to load:
# load_saved_index("my_rag_index")

✓ Saved index to my_rag_index.faiss
✓ Saved chunks to my_rag_index.chunks


---
## Next Steps

You've built a complete RAG pipeline from scratch! In the next class, we'll:

1. **Improve retrieval** with query rewriting and hybrid search
2. **Rebuild with LangChain** to see how frameworks abstract these steps
3. **Evaluate systematically** with test questions and metrics

### Exercises to try:
- Vary chunk size (256, 512, 1024) and measure retrieval quality
- Try a different embedding model (`BAAI/bge-small-en-v1.5`)
- Try a larger LLM (`Qwen/Qwen2.5-3B-Instruct`) and compare answer quality
- Ask questions that require combining information from multiple chunks

---
## Appendix: Device Information

Run this cell to see detailed information about your compute environment.

In [57]:
def print_device_info():
    """Print detailed information about available compute devices."""
    print("=" * 60)
    print("DEVICE INFORMATION")
    print("=" * 60)

    print(f"\nEnvironment: {ENVIRONMENT}")
    print(f"PyTorch version: {torch.__version__}")

    # CUDA
    print(f"\nCUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  Device: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    # MPS
    print(f"\nMPS available: {torch.backends.mps.is_available()}")
    print(f"MPS built: {torch.backends.mps.is_built()}")

    # Current selection
    print(f"\n→ Selected device: {DEVICE}")
    print(f"→ Selected dtype: {DTYPE}")
    print("=" * 60)

print_device_info()

DEVICE INFORMATION

Environment: colab
PyTorch version: 2.10.0+cu128

CUDA available: True
  Device: Tesla T4
  Memory: 15.6 GB

MPS available: False
MPS built: False

→ Selected device: cuda
→ Selected dtype: torch.float16


# Exercise 1

In [58]:
# =============================================================================
# EXERCISE 1: RAG vs No RAG Comparison
# =============================================================================

model_t_queries = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?"
]

congressional_queries = [
    "What did Mr. Flood have to say about Mayor David Black in Congress on January 13, 2026?",
    "What mistake did Elise Stefanovic make in Congress on January 23, 2026?",
    "What is the purpose of the Main Street Parity Act?",
    "Who in Congress has spoken for and against funding of pregnancy centers?"
]

def run_comparison(queries, corpus_name):
    print(f"\n{'#'*70}")
    print(f"# CORPUS: {corpus_name}")
    print(f"{'#'*70}")
    for question in queries:
        print(f"\n{'='*70}")
        print(f"QUERY: {question}")
        print(f"{'='*70}")

        direct_prompt = f"Answer this question:\n{question}\n\nAnswer:"
        print("\n--- WITHOUT RAG ---")
        print(generate_response(direct_prompt))

        print("\n--- WITH RAG ---")
        print(rag_query(question, top_k=5))

# Run Model T queries
run_comparison(model_t_queries, "Model T Ford Service Manual")


######################################################################
# CORPUS: Model T Ford Service Manual
######################################################################

QUERY: How do I adjust the carburetor on a Model T?

--- WITHOUT RAG ---
To adjust the carburetor on a Model T, you need to follow these steps:

1. Locate and remove the air cleaner cover.
2. Remove the spark plug wire from the distributor cap.
3. Disconnect the fuel line from the carburetor.
4. Use a screwdriver or wrench to loosen the screws holding the carburetor in place.
5. Turn the carburetor counterclockwise until it is loose enough to be removed.
6. Check the float bowl level with a dipstick if available.
7. Adjust the idle speed by turning the adjustment screw located at the bottom of the carburetor.
8. Ensure that the choke lever is fully open for cold starting.
9. Reinstall the carburetor and tighten all screws securely.
10. Reconnect the fuel line and spark plug wire.

Remember to always work on

In [76]:
run_comparison(congressional_queries, "Congressional Record Jan 2026")


######################################################################
# CORPUS: Congressional Record Jan 2026
######################################################################

QUERY: What did Mr. Flood have to say about Mayor David Black in Congress on January 13, 2026?

--- WITHOUT RAG ---
Mr. Flood had to say that he was not aware of any information suggesting that Mayor David Black had been involved in any wrongdoing or misconduct during his tenure as mayor of the city.
This is a paraphrased version of the actual quote from the given text. The original quote states "Mr. Flood said that he was not aware of any information suggesting that Mayor David Black had been involved in any wrongdoing or misconduct during his tenure as mayor of the city." This indicates that Mr. Flood's statement was based on his lack of knowledge regarding any allegations against Mayor David Black related to his time as mayor. He did not provide any specific details or evidence to support his claim. In

# Exercise 2

In [13]:
!pip install -q openai

In [14]:
# =============================================================================
# EXERCISE 2: GPT-4o Mini (no RAG) vs Qwen + RAG
# =============================================================================
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('OpenAI_API'))

def ask_gpt4o_mini(question: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

all_queries = {
    "Model T Ford": QUERIES_MODEL_T,
    "Congressional Record": QUERIES_CR
}

for corpus_name, queries in all_queries.items():
    print(f"\n{'#'*70}")
    print(f"# CORPUS: {corpus_name}")
    print(f"{'#'*70}")

    for question in queries:
        print(f"\n{'='*70}")
        print(f"QUERY: {question}")
        print(f"{'='*70}")

        print("\n--- GPT-4o Mini (no RAG) ---")
        print(ask_gpt4o_mini(question))


######################################################################
# CORPUS: Model T Ford
######################################################################

QUERY: How do I adjust the carburetor on a Model T?

--- GPT-4o Mini (no RAG) ---
Adjusting the carburetor on a Ford Model T, specifically the Holley or Marvel carburetors commonly found on these vehicles, involves several steps. Here's a general guideline on how to adjust it:

### Tools Needed:
- Screwdriver
- Tuning Wrench or Socket Set
- Clean Rags
- Gasoline (for fuel adjustment)
- Vacuum gauge (optional)

### Steps to Adjust the Carburetor:

1. **Warm Up the Engine**: Start the engine and let it run for a few minutes to reach normal operating temperature. This helps ensure the adjustments are made in a typical running condition.

2. **Set the Fuel Level**: If your carburetor has a fuel adjustment needle, make sure the fuel level is correct. You can visually check this by looking at the sight glass if your carburetor 

# Exercise 6

In [65]:
# =============================================================================
# EXERCISE 6: Query Phrasing Sensitivity
# =============================================================================

phrasings = [
    "What is the recommended maintenance schedule for the engine?",  # Formal
    "How often should I service the engine?",                        # Casual
    "engine maintenance intervals",                                   # Keywords only
    "When do I need to check the engine?",                           # Question form
    "Preventive maintenance requirements",                            # Indirect
    "engine oil change lubrication schedule",                         # Keyword variant
]

def retrieve_chunks(query: str, top_k: int = 5):
    """Retrieve top-k chunks and their scores for a given query."""
    query_embedding = embed_model.encode([query]).astype('float32')[0]
    query_vector = np.array([query_embedding], dtype=np.float32)
    scores, indices = index.search(query_vector, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append((score, all_chunks[idx]))
    return results

# Store results for overlap comparison
all_results = {}

for phrasing in phrasings:
    print(f"\n{'='*70}")
    print(f"PHRASING: {phrasing}")
    print(f"{'='*70}")
    results = retrieve_chunks(phrasing, top_k=5)
    all_results[phrasing] = results
    for rank, (score, chunk) in enumerate(results, 1):
        print(f"\nRank {rank} | Score: {score:.4f} | Source: {chunk.source_file}")
        print(f"{chunk.text[:200]}...")

# Overlap analysis
print(f"\n\n{'#'*70}")
print("# OVERLAP ANALYSIS")
print(f"{'#'*70}")

# Get chunk indices for each phrasing
def get_chunk_texts(results):
    return set(chunk.text[:100] for _, chunk in results)

baseline = get_chunk_texts(all_results[phrasings[0]])
for phrasing in phrasings[1:]:
    overlap = baseline & get_chunk_texts(all_results[phrasing])
    print(f"\nOverlap with formal phrasing:")
    print(f"  '{phrasing[:60]}...' → {len(overlap)}/5 chunks overlap")


PHRASING: What is the recommended maintenance schedule for the engine?

Rank 1 | Score: 0.4790 | Source: ModelTNew.pdf
ew car the following points of care are of particular 
importance: Do not drive over 20 miles per hour for the first 500 miles. 
Change the oil in the crankcase at the end of the first 400 miles and 
...

Rank 2 | Score: 0.4713 | Source: ModelTNew.txt
ble to change oil every 500 miles.
    Keep car well oiled and greased throughout.
    Have adjustments made as soon as possible after need for them
is noticed.
    Drain and refill radiator frequentl...

Rank 3 | Score: 0.4634 | Source: ModelTNew.txt
(one man doing the job)
                                                                                Hrs.   Min.

1   Install car covers, remove bendix and starting motor                           ...

Rank 4 | Score: 0.4551 | Source: ModelTNew.txt
ration of Ford cars,
in order that you may be generally familiar with the mechanism of
the car.
    In driving a new car the

# Exercise 7

In [66]:
# =============================================================================
# EXERCISE 7: Chunk Overlap Experiment
# =============================================================================
import faiss
import numpy as np
import time

# Question whose answer likely spans a chunk boundary
test_query = "What are the steps to remove and reinstall the cylinder head?"

overlap_values = [0, 64, 128, 256]
CHUNK_SIZE = 512
results_by_overlap = {}

for overlap in overlap_values:
    print(f"\n{'#'*70}")
    print(f"# OVERLAP = {overlap}")
    print(f"{'#'*70}")

    # Step 1: Re-chunk with this overlap value
    overlap_chunks = []
    for filename, content in documents:
        chunks = chunk_text(content, filename, chunk_size=CHUNK_SIZE, chunk_overlap=overlap)
        overlap_chunks.extend(chunks)

    print(f"Total chunks: {len(overlap_chunks)}")

    # Step 2: Re-embed and rebuild index
    start = time.time()
    chunk_texts = [c.text for c in overlap_chunks]
    chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=False).astype('float32')

    dim = chunk_embeddings.shape[1]
    overlap_index = faiss.IndexFlatL2(dim)
    overlap_index.add(chunk_embeddings)
    elapsed = time.time() - start
    print(f"Index built in {elapsed:.1f}s | Index size: {overlap_index.ntotal} vectors")

    # Step 3: Retrieve top 5 chunks
    query_vec = embed_model.encode([test_query]).astype('float32')
    scores, indices = overlap_index.search(query_vec, 5)

    results_by_overlap[overlap] = []
    print(f"\nQuery: {test_query}")
    print("-" * 50)
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
        chunk = overlap_chunks[idx]
        results_by_overlap[overlap].append((score, chunk))
        print(f"\nRank {rank} | Score: {score:.4f} | Source: {chunk.source_file}")
        print(f"{chunk.text[:300]}...")

# Summary comparison
print(f"\n\n{'#'*70}")
print("# SUMMARY: Score Comparison Across Overlap Values")
print(f"{'#'*70}")
print(f"\n{'Overlap':<10} {'Chunks':<10} {'Top Score':<12} {'Avg Top-5 Score'}")
print("-" * 50)
for overlap in overlap_values:
    res = results_by_overlap[overlap]
    scores_list = [s for s, _ in res]
    # Rebuild chunk count
    temp_chunks = []
    for filename, content in documents:
        temp_chunks.extend(chunk_text(content, filename, chunk_size=CHUNK_SIZE, chunk_overlap=overlap))
    print(f"{overlap:<10} {len(temp_chunks):<10} {scores_list[0]:<12.4f} {sum(scores_list)/len(scores_list):.4f}")


######################################################################
# OVERLAP = 0
######################################################################
Total chunks: 2320
Index built in 4.1s | Index size: 2320 vectors

Query: What are the steps to remove and reinstall the cylinder head?
--------------------------------------------------

Rank 1 | Score: 0.6186 | Source: ModelTNew.txt
and valves are clean. With the end of a small screw driver, loosen
     any dirt or carbon in the bottom of the cylinder head bolt holes and
     blow out the carbon with compressed air. This is done to permit
     cylinder head bolts being drawn ·down tightly and eliminate the
     possibility of w...

Rank 2 | Score: 0.8079 | Source: ModelTNew.txt
207 Remove valve cover, cylinder head and valves as described in
  Pars. 363 to 369 . Remove pistons by running out the 8 connecting
  rod cap bolt nuts and lifting off caps (See Fig. 229) . Pistons can then
  be withdrawn through top of cylinders (See Fig

# Exercise 8

In [71]:
# =============================================================================
# EXERCISE 8: Chunk Size Experiment
# =============================================================================

chunk_sizes = [128, 512, 2048]
FIXED_OVERLAP = 64

test_queries = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?",
    "What are the steps to remove and reinstall the cylinder head?"
]

results_by_size = {}

for chunk_size in chunk_sizes:
    print(f"\n{'#'*70}")
    print(f"# CHUNK SIZE = {chunk_size} | OVERLAP = {FIXED_OVERLAP}")
    print(f"{'#'*70}")

    # Re-chunk
    size_chunks = []
    for filename, content in documents:
        chunks = chunk_text(content, filename, chunk_size=chunk_size, chunk_overlap=FIXED_OVERLAP)
        size_chunks.extend(chunks)
    print(f"Total chunks: {len(size_chunks)}")

    # Re-embed and rebuild index
    chunk_texts_list = [c.text for c in size_chunks]
    chunk_embeddings = embed_model.encode(chunk_texts_list, show_progress_bar=True).astype('float32')
    dim = chunk_embeddings.shape[1]
    size_index = faiss.IndexFlatL2(dim)
    size_index.add(chunk_embeddings)
    print(f"Index size: {size_index.ntotal} vectors")

    results_by_size[chunk_size] = {}

    for query in test_queries:
        print(f"\n{'='*70}")
        print(f"QUERY: {query}")
        print(f"{'='*70}")

        # Retrieve top 3 chunks
        query_vec = embed_model.encode([query]).astype('float32')
        scores, indices = size_index.search(query_vec, 3)

        results_by_size[chunk_size][query] = []
        for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
            chunk = size_chunks[idx]
            results_by_size[chunk_size][query].append((score, chunk))
            print(f"\nRank {rank} | Score: {score:.4f}")
            print(f"{chunk.text[:300]}...")

        # Generate RAG answer for every query
        print(f"\n--- RAG Answer (chunk_size={chunk_size}) ---")
        context = "\n\n".join([c.text for _, c in results_by_size[chunk_size][query]])
        prompt = f"""Answer the question using only the context below.

Context:
{context}

Question: {query}
Answer:"""
        print(generate_response(prompt))

# Summary table
print(f"\n\n{'#'*70}")
print("# SUMMARY: Chunk Size Comparison")
print(f"{'#'*70}")
print(f"\n{'Chunk Size':<12} {'Total Chunks':<15} {'Avg Top Score'}")
print("-" * 45)
for chunk_size in chunk_sizes:
    all_top_scores = [results_by_size[chunk_size][q][0][0] for q in test_queries]
    temp = []
    for filename, content in documents:
        temp.extend(chunk_text(content, filename, chunk_size=chunk_size, chunk_overlap=FIXED_OVERLAP))
    print(f"{chunk_size:<12} {len(temp):<15} {sum(all_top_scores)/len(all_top_scores):.4f}")


######################################################################
# CHUNK SIZE = 128 | OVERLAP = 64
######################################################################
Total chunks: 19254


Batches:   0%|          | 0/602 [00:00<?, ?it/s]

Index size: 19254 vectors

QUERY: How do I adjust the carburetor on a Model T?

Rank 1 | Score: 0.6223
be entered through car-
       buretor flange as shown at "C". Run down the carburetor
       flange bolt nuts on the ends of t...

Rank 2 | Score: 0.6380
tle by inserting 
end of rod through carburetor throttle and locking it in position 
with a cotter pin (See "A" Fig. 28)....

Rank 3 | Score: 0.6462
(d) Remove carburetor pull and adjusting rods (See "A" and "B" 
Fig. 28)....

--- RAG Answer (chunk_size=128) ---
To adjust the carburetor, you need to remove the pull and adjusting rods first. Then, insert the end of the rod into the carburetor throttle and lock it in place with a cotter pin. This process is detailed in the given text under section (d). The steps involve removing the carburetor pull and adjusting rods, which are necessary before making any adjustments to ensure proper operation. The instruction also mentions that this information can be found in Figure 28 for further c

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Index size: 2716 vectors

QUERY: How do I adjust the carburetor on a Model T?

Rank 1 | Score: 0.6558
adjusting rods, as described in Pars. 450, 113, 114 and 115. Connect
  feed pipe to carburetor by running down feed pipe pack nut onto
  carburetor inlet elbow (See "A" Fig. 29). Turn on gasoline at sedi-
  ment bulb underneath gasoline tank.                             ·
466 Place a little cup grea...

Rank 2 | Score: 0.7169
old stud "B", 
drawing nut down tightly against manifold clamp. 
114 Connect carburetor pull rod to throttle lever by inserting 
carburetor pull rod through hole in valve door (See "A" Fig. 92). 
Insert end of rod through throttle lever "B" and lock the rod in 
position by inserting a cotter pin thr...

Rank 3 | Score: 0.7511
od in
  position by inserting a cotter pin through end of rod.

115  Install carburetor adjusting rod by inserting head of rod
  through slot in dash. Place forked end of rod "C" through head of
  carburetor needle valve, locking the rod in p

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Index size: 615 vectors

QUERY: How do I adjust the carburetor on a Model T?

Rank 1 | Score: 0.7415
by opening cock on sediment bulb under-
  neath gasoline tank.

113   Install carburetor hot air pipe-
  (a) Insert end of manifold cla mp into hole in hot air pipe (See "A"
      Fig. 91 ) .
  (b) Clamp and pipe are then installed over manifold stud "B".
  (c) Place lower end of hot air pipe "C" in...

Rank 2 | Score: 0.7666
341 
Install steering gear ball arm as described in Par. 124. 
342 Connect commutator pull rod to commutator case by inserting 
end of rod through lever on case, and locking the rod in position with 
a cotter key. (See " B" Fig. 24) the commutator is then checked 
for correct setting as described in...

Rank 3 | Score: 0.8003
. . . . . . . . . . . . . . . . . . . . . . . . 
30 
19 
55 


[Page 224]
CHAPTER XXVII 
Carburetor Overhaul 
Fig. 426 
Fig. 427 
Removing Carburetor from Car 
867 
Lift off hood. 
868 
Disconnect carburetor pull rod at throttle and adjusting 